# One CNN Backbone, Two Computer Vision Tasks

## WORKED SOLUTION

### Part 0 — What Are We Building?

We will train two small models from scratch using real cat and dog photographs:

1. **Image classification:** decide whether the whole image shows a cat or a dog.
2. **Binary semantic segmentation:** decide whether each pixel belongs to the animal or the background.

Both models use the same **backbone architecture**. A backbone is the part of a neural network that turns pixels into useful **feature maps**—grids of learned visual patterns. Each task then adds its own output **head**, which converts those features into the required prediction.

```text
Input image: [3, 96, 96]
            │
            ▼
SharedBackbone architecture
            │
            ▼
Feature maps: [64, 24, 24]
            │
      ┌─────┴──────────────┐
      ▼                    ▼
Classification head   Segmentation head
      │                    │
      ▼                    ▼
Cat / dog scores      Animal/background
for one image         score for every pixel
```

We create **two separate models**. They reuse the backbone design, but they do not share learned weights. Both begin with random weights and train independently.

### Learning objectives

By the end, you should be able to:

- Explain how a CNN backbone extracts visual features.
- Trace how tensor shapes change through convolution and pooling layers.
- Compare an image-level label with a pixel-level segmentation mask.
- Explain why classification and segmentation need different heads and losses.
- Adapt one backbone architecture to two computer vision tasks.

### Workshop route

| Part | Activity |
| --- | --- |
| 0 | Understand the two tasks and set up the notebook |
| 1 | Inspect pet images, class labels and masks |
| 2 | Build and inspect the shared CNN backbone |
| 3 | Train a cat-versus-dog classifier |
| 4 | Train an animal-versus-background segmenter |
| 5 | Compare what stays the same and what changes |
| 6 | Review the key ideas |

The notebook uses a prepared **10 MB** subset of Oxford-IIIT Pet included in `data/pet_mini.npz`.

### 0.1 Setup

**What are we doing?** Importing the libraries, setting random seeds and selecting the fastest available device.

**Why are we doing it?** PyTorch performs the tensor operations and model training. Fixed seeds make the data selection and starting weights more reproducible.

**Code:** Run the installation cell only if an import fails. Keep `requirements.txt` in the notebook's working directory.

In [ ]:
# Install into the currently selected notebook kernel if needed.
# %pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
SIZE, CLASSES = 96, ["cat", "dog"]
print("Device:", device)

**What should we observe?** The cell prints `cuda`, `mps` or `cpu`, followed by “Setup successful!”. An **epoch** later means one complete pass through the training examples.

## Part 1 — Understanding Our Dataset

We use the [Oxford-IIIT Pet dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/), which contains pet photographs and human-created segmentation annotations.

**What are we doing?** Preparing the same photographs with two kinds of targets:

- An image-level label: `0 = cat`, `1 = dog`.
- A pixel-level mask: `0 = background`, `1 = animal`.

**Why are we doing it?** The target tells a model what question to learn. One number describes the whole image, while a mask supplies one answer at every pixel.

The original annotation also marks an animal boundary. We combine the animal and boundary into one foreground class. Images use bilinear resizing; masks use nearest-neighbour resizing so label values stay discrete.

We use a balanced prepared subset: 320 training images, 64 validation images and 64 held-out test images. Validation data monitors learning. Test data is used only after training.

**Dataset credit:** O. M. Parkhi, A. Vedaldi, A. Zisserman and C. V. Jawahar, *Cats and Dogs*, CVPR 2012. The dataset is distributed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/); image copyrights remain with their original owners.

### 1.1 Prepare the data

**Code:** The compact file already contains 96 × 96 images and masks; this cell converts them into PyTorch tensors.

In [ ]:
data = np.load("data/pet_mini.npz")


def load_split(name):
    images = torch.from_numpy(data[f"{name}_images"]).permute(0, 3, 1, 2).float() / 255
    labels = torch.from_numpy(data[f"{name}_labels"]).long()
    masks = torch.from_numpy(data[f"{name}_masks"]).unsqueeze(1).float()
    return images, labels, masks


X_train, y_train, masks_train = load_split("train")
X_valid, y_valid, masks_valid = load_split("valid")
X_test, y_test, masks_test = load_split("test")


def loader(images, targets, shuffle=False):
    return DataLoader(TensorDataset(images, targets), 32, shuffle=shuffle)


class_train = loader(X_train, y_train, True)
class_valid = loader(X_valid, y_valid)
class_test = loader(X_test, y_test)
seg_train = loader(X_train, masks_train, True)
seg_valid = loader(X_valid, masks_valid)
seg_test = loader(X_test, masks_test)

print("Train:", X_train.shape, y_train.shape, masks_train.shape)
print("Validation:", len(X_valid), "| Test:", len(X_test))

**What should we observe?**

| Tensor | Shape | Meaning |
| --- | --- | --- |
| Training images | `[320, 3, 96, 96]` | 320 RGB training images |
| Classification labels | `[320]` | One cat/dog label per training image |
| Segmentation masks | `[320, 1, 96, 96]` | One binary mask per training image |

The leading number counts examples. The image channels are red, green and blue; the mask has one channel.

### 1.2 Compare an image, its label and its mask

**What are we doing?** Displaying one dog and one cat with their labelled masks.

**Why are we doing it?** This makes the two learning targets visible before we build either model.

In [ ]:
indices = [(y_train == label).nonzero()[0, 0] for label in (1, 0)]
fig, axes = plt.subplots(2, 2, figsize=(8, 7))
for row, i in enumerate(indices):
    axes[row, 0].imshow(X_train[i].permute(1, 2, 0))
    axes[row, 0].set_title(CLASSES[y_train[i]])
    axes[row, 1].imshow(masks_train[i, 0], cmap="gray")
    axes[row, 1].set_title("Animal mask")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

**What should we observe?** White mask pixels belong to the animal and black pixels are background. The binary mask locates the animal but does not say cat or dog.

**Check your understanding:** For the dog photograph, which target answers “what is it?” and which target answers “where is it?”

## Part 2 — Building Our CNN Backbone

A **convolutional layer** slides small learned filters across an image. Its outputs are **feature maps**: spatial grids that respond to useful patterns.

- **Convolution (`Conv2d`)** learns local patterns and changes the number of channels.
- **ReLU** keeps positive responses and adds non-linearity, allowing richer patterns.
- **Pooling (`MaxPool2d`)** halves height and width while retaining strong responses.
- **Padding** adds pixels around the input so a 3 × 3 convolution can preserve spatial size.

### 2.1 See how padding changes a feature map

**What are we doing?** Applying one 3 × 3 convolution with three padding values.

**Why are we doing it?** Understanding spatial dimensions makes the larger backbone easier to follow.

For stride 1, padding 0 changes 96 × 96 to 94 × 94; padding 1 preserves 96 × 96.

In [ ]:
example = X_train[:1]
for padding in (0, 1, 2):
    output = nn.Conv2d(3, 8, 3, padding=padding)(example)
    print(f"padding={padding}: {tuple(example.shape)} -> {tuple(output.shape)}")

**What should we observe?** The outputs are `(1, 8, 94, 94)`, `(1, 8, 96, 96)` and `(1, 8, 98, 98)`. The `8` comes from `out_channels=8`.

### 2.2 Define the shared backbone

**What are we doing?** Stacking six convolution layers in three stages.

**Why are we doing it?** The backbone converts pixels into learned features. It deliberately stops before making a task-specific prediction.

**Code:** This class is the central architecture reused by both models.

In [ ]:
def conv_stage(in_channels, out_channels, pool=False):
    layers = [
        nn.Conv2d(in_channels, out_channels, 3, padding=1), nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, 3, padding=1), nn.ReLU(),
    ]
    return nn.Sequential(*layers, *([nn.MaxPool2d(2)] if pool else []))


class SharedBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.stage1 = conv_stage(3, 16, pool=True)
        self.stage2 = conv_stage(16, 32, pool=True)
        self.stage3 = conv_stage(32, 64)

    def forward(self, x):
        return self.stage3(self.stage2(self.stage1(x)))


backbone = SharedBackbone()
a = backbone.stage1(example)
b = backbone.stage2(a)
c = backbone.stage3(b)
print("Input:", example.shape, "→ Stage 1:", a.shape,
      "→ Stage 2:", b.shape, "→ Stage 3:", c.shape)

**What should we observe?**

| Point in network | Tensor shape | Change |
| --- | --- | --- |
| Input | `[1, 3, 96, 96]` | One RGB photograph |
| Stage 1 | `[1, 16, 48, 48]` | More channels; first pooling halves space |
| Stage 2 | `[1, 32, 24, 24]` | More channels; second pooling halves space |
| Stage 3 | `[1, 64, 24, 24]` | More features; spatial size is retained |

As the network becomes deeper, spatial dimensions shrink while the number of feature channels grows. The final `[64, 24, 24]` feature maps still retain locations needed for segmentation.

## Part 3 — Task A: Image Classification

**What are we doing?** Predicting one label—cat or dog—for each photograph.

**Why are we doing it?** Classification needs a single image-level answer. `AdaptiveAvgPool2d(1)` summarises each of the 64 feature maps, and `Linear(64, 2)` produces two **logits**.

A **logit** is a raw model score before conversion to a probability. `CrossEntropyLoss` compares the two logits with the correct integer class label. A **loss** is a number measuring prediction error; training changes model weights to reduce it.

### 3.1 Define and inspect the classifier

**Code:** Notice that `CNNClassifier` contains a `SharedBackbone` followed by a classification head.

In [ ]:
class CNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = SharedBackbone()
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.head(self.backbone(x))


classifier = CNNClassifier().to(device)
print(classifier)

**What should we observe?** The printed model has two visible parts: `backbone` and `head`. The head ends with two outputs because there are two classes.

### 3.2 Train the classifier

**What are we doing?** Repeatedly predicting, measuring cross-entropy loss, computing gradients and updating the weights.

**Why are we doing it?** The randomly initialised filters must learn patterns that help distinguish cats from dogs.

**Code:** The same general training loop will also train segmentation; only the loaders and loss function change.

In [ ]:
def average_loss(model, data_loader, loss_fn):
    model.eval()
    with torch.no_grad():
        losses = [loss_fn(model(x.to(device)), y.to(device)).item()
                  for x, y in data_loader]
    return sum(losses) / len(losses)


def fit(model, train_loader, valid_loader, loss_fn, epochs=10):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        print(f"Epoch {epoch:2}: train={sum(losses)/len(losses):.3f} | "
              f"valid={average_loss(model, valid_loader, loss_fn):.3f}")
    return model


classifier = fit(classifier, class_train, class_valid,
                 nn.CrossEntropyLoss(), epochs=10)

**What should we observe?** Training and validation loss should generally decrease, although they may fluctuate. One epoch is one full pass through the training subset. Exact results vary by device and run.

### 3.3 Evaluate and view predictions

**What are we doing?** Predicting labels for held-out photographs and calculating accuracy.

**Why are we doing it?** **Accuracy** is the fraction of test images assigned the correct cat/dog label. Held-out images give a fairer check than training images.

In [ ]:
classifier.eval()
images, labels = next(iter(class_test))
with torch.no_grad():
    logits = classifier(images.to(device)).cpu()
predictions = logits.argmax(1)

correct = 0
with torch.no_grad():
    for x, y in class_test:
        correct += (classifier(x.to(device)).argmax(1).cpu() == y).sum().item()
print(f"Test accuracy: {correct / len(class_test.dataset):.1%}")

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, image, actual, predicted in zip(axes, images, labels, predictions):
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title(f"True: {CLASSES[actual]}\nPred: {CLASSES[predicted]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

**What should we observe?** Each output row contains two logits. `argmax(dim=1)` selects the larger score. The plot shows the true and predicted label for the same fixed examples, whether the predictions are correct or incorrect.

This model starts from random weights and sees a small subset, so it may perform much worse than the pretrained MobileNetV2 used earlier.

## Part 4 — Task B: Image Segmentation

Classification asks **“cat or dog?”** Segmentation asks **“which pixels belong to the animal?”**

**What are we doing?** Reusing the `SharedBackbone` architecture with a new segmentation head.

**Why are we doing it?** Segmentation needs a spatial output. We keep the 24 × 24 feature grid instead of collapsing it:

- A **1 × 1 convolution** converts the 64 features at every location into one animal/background logit.
- **Bilinear upsampling** resizes the 24 × 24 logit grid back to 96 × 96.
- `BCEWithLogitsLoss` compares each raw pixel logit with a 0/1 target mask.

### 4.1 Define, inspect and train the segmenter

**Code:** This creates a fresh backbone instance with new random weights. It uses the same backbone design, not the classifier's trained weights.

In [ ]:
class CNNSegmenter(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = SharedBackbone()  # Fresh weights.
        self.head = nn.Sequential(
            nn.Conv2d(64, 1, 1),
            nn.Upsample((SIZE, SIZE), mode="bilinear", align_corners=False),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


segmenter = CNNSegmenter().to(device)
print(segmenter)
segmenter = fit(segmenter, seg_train, seg_valid,
                nn.BCEWithLogitsLoss(), epochs=10)

**What should we observe?** The printed model contains the same `SharedBackbone` architecture but a different head. Training again begins from random weights and uses a pixel-level loss.

### 4.2 Evaluate and view predicted masks

**What are we doing?** Applying a sigmoid to turn logits into scores from 0 to 1, then using 0.5 as the animal/background threshold.

**Why are we doing it?** We compare predicted masks with ground truth using **Intersection over Union (IoU)**:

`IoU = overlapping animal pixels / pixels labelled as animal by either mask`

IoU focuses on foreground overlap. Pixel accuracy can look high simply because much of an image is background.

In [ ]:
segmenter.eval()
images, true_masks = next(iter(seg_test))
with torch.no_grad():
    predicted_masks = segmenter(images.to(device)).sigmoid().cpu() >= 0.5

intersection = union = 0
with torch.no_grad():
    for x, y in seg_test:
        prediction = segmenter(x.to(device)).sigmoid().cpu() >= 0.5
        truth = y.bool()
        intersection += (prediction & truth).sum().item()
        union += (prediction | truth).sum().item()
print(f"Test IoU: {intersection / union:.3f}")

fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for row in range(3):
    axes[row, 0].imshow(images[row].permute(1, 2, 0))
    axes[row, 1].imshow(true_masks[row, 0], cmap="gray")
    axes[row, 2].imshow(predicted_masks[row, 0], cmap="gray")
for ax, title in zip(axes[0], ["Image", "True mask", "Predicted mask"]):
    ax.set_title(title)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

**What should we observe?** The columns show the photograph, ground-truth mask and predicted mask. Boundaries may look coarse because pooling reduces the feature maps to 24 × 24 before upsampling. Bilinear upsampling enlarges predictions; it does not restore detail that was lost.

This is binary semantic segmentation: all animal pixels share one foreground class. It does not separate multiple animal instances.

## Part 5 — Comparing Both Tasks

The image input and backbone architecture stay the same. The task-specific head, prediction target, loss and evaluation measure change.

| Component | Classification | Segmentation |
| --- | --- | --- |
| Input | Image `[B, 3, 96, 96]` | Image `[B, 3, 96, 96]` |
| Backbone | `SharedBackbone` architecture | `SharedBackbone` architecture |
| Backbone output | `[B, 64, 24, 24]` | `[B, 64, 24, 24]` |
| Output head | Global pooling + linear layer | 1 × 1 convolution + upsampling |
| Prediction | One cat/dog label per training image | One animal/background label per pixel |
| Output logits | `[B, 2]` | `[B, 1, 96, 96]` |
| Target | Integer class label `[B]` | Float mask `[B, 1, 96, 96]` |
| Loss | `CrossEntropyLoss` | `BCEWithLogitsLoss` |
| Evaluation | Accuracy | Foreground IoU |

```text
What stays the same: image format + SharedBackbone class + general training steps
What changes:         output head + target labels + loss + evaluation
```

Each model owns a separate `SharedBackbone()` instance. Training the classifier does not update the segmenter, and training the segmenter does not update the classifier.

### 5.1 Apply both tasks to the workshop Labrador

**What are we doing?** Running both independently trained models on the same Labrador photograph used in the introductory notebook.

**Why are we doing it?** The side-by-side output makes the distinction concrete: classification returns one label, while segmentation returns a mask.

There is no ground-truth mask for this photograph here, so we display the segmentation prediction without calculating IoU.

**Image credit:** *YellowLabradorLooking new.jpg*, original photo by Elf, derivative/retouch by Djmirko and FT2, via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:YellowLabradorLooking_new.jpg), [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/).

In [ ]:
photo_path = Path("data/YellowLabradorLooking_new.jpg")
if not photo_path.exists():
    urlretrieve(
        "https://storage.googleapis.com/download.tensorflow.org/example_images/"
        "YellowLabradorLooking_new.jpg", photo_path
    )
with Image.open(photo_path) as file:
    photo = ImageOps.exif_transpose(file).convert("RGB")
array = np.asarray(photo.resize((SIZE, SIZE)), dtype=np.float32) / 255
small = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0).to(device)

classifier.eval()
segmenter.eval()
with torch.no_grad():
    label = classifier(small).argmax(1).item()
    mask = segmenter(small).sigmoid()[0, 0].cpu() >= 0.5

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(photo)
axes[0].set_title(f"Classification: {CLASSES[label]}")
axes[1].imshow(photo.resize((SIZE, SIZE)))
axes[1].imshow(np.ma.masked_where(~mask, mask), cmap="autumn", alpha=0.45)
axes[1].set_title("Segmentation mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

**What should we observe?** The classifier produces one cat/dog decision. The segmenter highlights pixels it considers part of the animal. Either result may be imperfect because these compact models trained briefly from scratch.

## Part 6 — Key Takeaways

1. A CNN backbone learns feature maps from image pixels.
2. Classification and segmentation can use the same backbone architecture.
3. Classification collapses spatial features to predict one class per image.
4. Segmentation preserves or reconstructs spatial information to predict one label per pixel.
5. Different targets require suitable heads, losses and evaluation measures.
6. Architecture reuse does not imply weight sharing: our two models trained independently from scratch.

### Check your understanding

1. Why would global average pooling be useful for classification but unsuitable as the final step for segmentation?
2. If the segmentation head outputs `[B, 1, 96, 96]`, what shape should its target mask have?
3. Which components would you change to adapt this backbone to a new pixel-level task?

### Connection to the next activity

The hand-gesture notebook uses a different pipeline. A pretrained MediaPipe model extracts hand landmarks, then a classifier learns from those landmark features rather than directly from image pixels.